In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.gridspec as gs
import regionmask
import xarray as xr
import cartopy.crs as ccrs
import seaborn as sns
from shapely.geometry import box
from shapely.geometry import Polygon
from shapely.geometry import Point

# Load Data

In [ ]:
# load fire observations
fire_obs = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Fire_Observations/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations.shp")

# load study area
study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")

# Regionalization (10 regions)

In [ ]:
# British Isles (BI)
BI = box(-11, 50, 1.5, 65)

# Iberian Peninsula (IP)
IP = box(-10, 34, 5, 44)

# France (FR)
FR = box(-5, 44, 5, 50)

# Mid Europe (ME)
ME = Polygon([(1.5, 50), (1.5, 55), (19, 55), (19, 50), (15, 50), (15, 48), (5, 48), (5, 50)])

# The alps (AL)
AL = box(5, 44, 15, 48)

# South Eastern Europe (SEA)
SEA = box(15, 44, 32, 50)

# North Eastern Europe (NEA)
NEA = box(19, 50, 32, 60)

# Scandinavia (SC)
SC = Polygon([(4, 55), (4, 71.5), (32, 71.5), (32, 60), (19, 60), (19, 55)])

# West Mediterranean (WMD)
WMD = Polygon([(5, 44), (5, 34), (19, 34), (19, 40), (15, 44)])

# East Mediterranean (EMD)
EMD = Polygon([(15, 44), (32, 44), (32, 34), (19, 34), (19, 40)])

In [ ]:
region_ten = gpd.GeoDataFrame({'geometry': [BI, IP, FR, ME, AL, SEA, NEA, SC, WMD, EMD],
                               'region': ["British Isles", "Iberian Peninsula", "France", "Mid Europe", "The alps", "South Eastern Europe", "North Eastern Europe", "Scandinavia", "West Mediterranean", "East Mediterranean"],
                               'abbrev': ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]
                              }, 
                               crs = "EPSG:4326")

In [ ]:
# get fire centers
fire_obs = fire_obs.to_crs(epsg = 3035)
fire_centers = list(zip(fire_obs.geometry.centroid.x, fire_obs.geometry.centroid.y))
gpd_fire_centers = gpd.GeoDataFrame(geometry = [Point(fire_center) for fire_center in fire_centers], crs = "EPSG:3035")
gpd_fire_centers = gpd_fire_centers.to_crs(epsg = 4326)
fire_obs = fire_obs.to_crs(epsg = 4326)

In [ ]:
# plot study area, fire events and 10 regions
plt.figure(figsize = (20, 12))
ax = plt.gca()

#----------------------
study_area.boundary.plot(ax = ax, edgecolor = "#909090", linewidth = 1)

#----------------------
gpd_fire_centers.plot(ax = ax, markersize = 0.5, marker='o',  color = "red")

#----------------------
region_ten.boundary.plot(ax = ax, edgecolor = "black")

#----------------------
# add region labels
ax.text(-10, 64, "BI", fontsize = 15)
ax.text(-9, 43, "IP", fontsize = 15)
ax.text(-4, 49, "FR", fontsize = 15)
ax.text(2.5, 54, "ME", fontsize = 15)
ax.text(5, 70.5, "SC", fontsize = 15)
ax.text(6, 47, "AL", fontsize = 15)
ax.text(6, 43, "WMD", fontsize = 15)
ax.text(16.5, 43, "EMD", fontsize = 15)
ax.text(20, 59, "NEA", fontsize = 15)
ax.text(16.5, 49, "SEA", fontsize = 15)

plt.show()

In [ ]:
region_ten

In [ ]:
region_list = ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]

# check each region individually
fig, axes = plt.subplots(2, 5, figsize = (25, 10))
axes = axes.flatten()

for reg, ax in zip(region_list, axes):
    region_shp = region_ten[region_ten["abbrev"] == reg]
    study_area.boundary.plot(ax = ax, edgecolor = "#909090", linewidth = 1)
    region_shp.boundary.plot(ax = ax, edgecolor = "red", linewidth = 1)

    ax.set_aspect(1.5, adjustable='box')
    
plt.tight_layout()
plt.subplots_adjust(hspace=0.1, wspace=0.1)
plt.show()

In [ ]:
region_ten.to_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp")

# Assign each fire to a region

In [ ]:
#label the region each fire event belongs to
fire_obs["region"] = pd.Series(np.nan, dtype = "str")

In [ ]:
for index, fire in fire_obs.iterrows():
    
    fire_geometry = gpd.GeoSeries(data = [fire.geometry], crs = "EPSG:4326")
    fire_geometry = fire_geometry.to_crs(epsg = 3035)
    
    #get fire centroid in 3035 prj
    fire_centroid = zip(fire_geometry.geometry.centroid.x, fire_geometry.geometry.centroid.y)

    #reproject the centroid to 4326 prj
    gdf_fire_centroid = gpd.GeoDataFrame({'geometry': [Point(fire_centroid)]}, crs = "EPSG:3035")
    gdf_fire_centroid = gdf_fire_centroid.to_crs(epsg = 4326)

    for _, box in region_ten.iterrows():
        bounding_box = box.geometry
        if gdf_fire_centroid.geometry.iloc[0].covered_by(bounding_box):  #both inside and on the border
            fire_obs.loc[index, "region"] = box["abbrev"]
            break

In [ ]:
# check NA
fire_obs["region"].isna().any()

In [ ]:
# regional fire event numbers (sum up to 20103)
fire_obs["region"].value_counts()

In [ ]:
# visual check
region_list = ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]

# check each region individually
fig, axes = plt.subplots(2, 5, figsize = (25, 10))
axes = axes.flatten()

for reg, ax in zip(region_list, axes):

    fire_obs_region = fire_obs[fire_obs["region"] == reg]

    # reproject 3035
    fire_obs_region = fire_obs_region.to_crs(epsg = 3035)

    # get centers
    fire_obs_centers = zip(fire_obs_region.geometry.centroid.x, fire_obs_region.geometry.centroid.y)
    fire_obs_centers_gdf = gpd.GeoDataFrame({'geometry': [Point(fire_center) for fire_center in fire_obs_centers]}, crs = "EPSG:3035")
    fire_obs_centers_gdf = fire_obs_centers_gdf.to_crs(epsg = 4326)
    
    region_shp = region_ten[region_ten["abbrev"] == reg]
    
    study_area.boundary.plot(ax = ax, edgecolor = "#909090", linewidth = 1)
    region_shp.boundary.plot(ax = ax, edgecolor = "black", linewidth = 1)
    fire_obs_centers_gdf.plot(ax = ax, marker = 'o', color = "red", markersize = 0.8)

    ax.set_aspect(1.5, adjustable='box')
    
plt.tight_layout()
plt.subplots_adjust(hspace=0.1, wspace=0.1)
plt.show()

In [ ]:
fire_obs.to_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region.shp")

# Fire Seasonality

In [ ]:
fire_seas = pd.DataFrame(columns = ["Region", "Month", "EV", "BA"]) #region, month, number of fire event, cumulative burned area
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

In [ ]:
for reg in region_list:
    
    fire_obs_reg = fire_obs[fire_obs["region"] == reg].copy()
    fire_obs_reg["start_date"] = pd.to_datetime(fire_obs_reg["start_date"])
    fire_seas_reg = pd.DataFrame(columns = ["Region", "Month", "EV", "BA"])
    fire_seas_reg["Region"] = [reg for _ in range(12)]
    fire_seas_reg["Month"] = months
    
    for ind, mon in enumerate(months):
        
        fire_obs_reg_mon = fire_obs_reg[fire_obs_reg['start_date'].dt.month == ind+1]
        
        #avg number of events across the 20 yrs period
        EV = len(fire_obs_reg_mon)/20
        
        #avg cumulative burned area across the 20 yrs period
        BA = fire_obs_reg_mon["area"].sum()/20

        fire_seas_reg.loc[fire_seas_reg["Month"] == mon, "EV"] = EV
        fire_seas_reg.loc[fire_seas_reg["Month"] == mon, "BA"] = BA

    # rescaled to monthly maximal values
    fire_seas_reg["EV"] = fire_seas_reg["EV"]/(fire_seas_reg["EV"].max())
    fire_seas_reg["BA"] = fire_seas_reg["BA"]/(fire_seas_reg["BA"].max())

    fire_seas = pd.concat([fire_seas, fire_seas_reg], ignore_index = True)

In [ ]:
#visualize seasonality
fig, axes = plt.subplots(2, 5, figsize=(24, 10), subplot_kw=dict(polar=True))

for (ax, reg) in zip(axes.flatten(), ["BI", "ME", "NEA", "IP", "SC", "WMD", "EMD", "AL", "SEA", "FR"]):
    
    #get data and angles
    fire_seas_reg = fire_seas[fire_seas["Region"] == reg]
    num_ev = list(fire_seas_reg["EV"])
    burned_area = list(fire_seas_reg["BA"])
    angles = np.linspace(0, 2 * np.pi, len(months), endpoint=False).tolist()
    
    # close the loop, append the first to the end of the list
    num_ev += num_ev[:1]
    burned_area += burned_area[:1]
    angles += angles[:1]
    
    # Draw one axe per variable and add labels
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1], months, fontsize = 12)
    ax.tick_params(axis='x', pad=8)
    
    # Draw ylabels
    ax.set_rlabel_position(0)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0], ["0", "0.2", "0.4", "0.6", "0.8", "1.0"], color="black", fontsize=12)
    ax.set_ylim(0, 1)
    
    # Plot data
    ax.plot(angles, num_ev, color='red', linewidth=2, linestyle='solid', label='FO')
    ax.fill(angles, num_ev, color='red', alpha=0.25)
    
    ax.plot(angles, burned_area, color='blue', linewidth=2, linestyle='solid', label='BA')
    ax.fill(angles, burned_area, color='blue', alpha=0.25)
    ax.set_title(reg, fontsize = 18)
    
    
# Add legend
plt.legend(loc = "lower center", bbox_to_anchor=(-2.1, -0.3), frameon = False, ncol = 2, fontsize = 16)

# Show the plot
plt.subplots_adjust(wspace=0.3, hspace=0.3)

In [ ]:
fire_seas.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Fire_Seasonality.csv", index = False)

# Fire Interannual Variability

In [ ]:
#initialize dataframe
interannual_fire = pd.DataFrame(columns = ["yr", "BA", "FO", "region"])   #year, cumulative burned area, fire occurrences, region
interannual_fire["yr"] = [yr for yr in range(2001, 2021)] * len(region_list)
interannual_fire["region"] = [reg for reg in region_list for _ in range (20)]

In [ ]:
for reg in region_list:
    
    fire_obs_reg = fire_obs[fire_obs["region"] == reg].copy()
    
    for yr in range(2001, 2021):
        
        fire_obs_reg["start_date"] = pd.to_datetime(fire_obs_reg["start_date"])
        fire_obs_reg_yr = fire_obs_reg[fire_obs_reg["start_date"].dt.year == yr]
        BA = fire_obs_reg_yr["area"].sum() * 100   #convert km2 to hectare
        FO = len(fire_obs_reg_yr)
        
        interannual_fire.loc[(interannual_fire["yr"] == yr) & (interannual_fire["region"] == reg), "BA"] = BA
        interannual_fire.loc[(interannual_fire["yr"] == yr) & (interannual_fire["region"] == reg), "FO"] = FO

In [ ]:
interannual_fire.to_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Fire_Interannual_Variability.csv", index = False)

In [ ]:
interannual_fire = pd.read_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Fire_Interannual_Variability.csv")
region_list = ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]
region_lb = ["BI", "IP", "FR", "CE", "AL", "SEE", "NEE", "SC", "WMD", "EMD"]

#visualization
fig, axes = plt.subplots(5,2, figsize = (15, 13))
axes = axes.flatten()

for reg, ax in zip(region_list, axes):
    
    interannual_fire_reg = interannual_fire[interannual_fire["region"] == reg]

    n_total = sum(interannual_fire_reg["FO"])
    # Plot Variable1 as bars
    ax.bar(interannual_fire_reg['yr'], interannual_fire_reg['BA'], color='#275dc5')
    ax.set_xlabel('')
    ax.set_xticks(range(2001, 2021))
    ax.set_xticklabels(range(2001, 2021), rotation=45, ha="right")
    ax.set_ylabel('Burned area (ha)', color='#275dc5', fontsize = 13)
    ax.tick_params(axis='y', labelcolor='#275dc5')
    ax.set_title(f'{region_lb[region_list.index(reg)]} (n = {n_total})', fontsize = 17)
    
    # Create a secondary y-axis
    ax2 = ax.twinx()
    
    # Plot Variable2 as dots
    ax2.plot(interannual_fire_reg['yr'], interannual_fire_reg['FO'], color='black', marker='o')
    ax2.set_ylabel('Event number (-)', color='black', fontsize = 13)
    ax2.tick_params(axis='y', labelcolor='black')
    ax2.set_ylim(bottom=0)


plt.tight_layout()

plt.savefig("/home/lixinh/SynFire_Scripts/WP1/Figures/R_CEE_fire_interannual_variability.png", dpi = 600, bbox_inches = 'tight')

# Create Region Masks

In [ ]:
region_land = gpd.GeoDataFrame({"geometry": [gpd.clip(study_area, geom.geometry).unary_union for _, geom in region_ten.iterrows()],
                                "region": region_ten["region"],
                                "abbrev": region_ten["abbrev"]},
                               crs = "EPSG:4326")

In [ ]:
land_mask_binary = xr.open_dataarray("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Land_Mask_Binary_CERRA_reproject_EPSG4326_study_area_32E.nc")

In [ ]:
region_mask = regionmask.mask_geopandas(region_land, land_mask_binary.x.values, land_mask_binary.y.values)
region_mask = region_mask.rename({"lat": "y", "lon": "x"})

In [ ]:
f, ax = plt.subplots(subplot_kw=dict(projection=ccrs.PlateCarree()))
region_mask.where(region_mask == 6).plot(   # subset region based on the index in the region_ten
    ax=ax,
    transform=ccrs.PlateCarree(),
    add_colorbar=False,
)
ax.coastlines(color="0.1")

In [ ]:
# save
region_mask.to_netcdf("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Region_Mask_CERRA_reproject_EPSG4326_Ten.nc")

# Plot

In [ ]:
#load fire seasonality
fire_seas = pd.read_csv("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Fire_Seasonality.csv")

#load ten regions
region_ten = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp")

#load fire obs (with regions)
fire_obs = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region.shp")

#load study area
study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")

In [ ]:
#--------------------------------------------------------
#plot layout
introfig = plt.figure(figsize = (20, 16))
grid = gs.GridSpec(3, 4, figure = introfig)

#radar plots
SC = introfig.add_subplot(grid[0, 0], polar = True)
WMD = introfig.add_subplot(grid[0, 1], polar = True)
EMD = introfig.add_subplot(grid[0, 2], polar = True)
IP = introfig.add_subplot(grid[0, 3], polar = True)

ME = introfig.add_subplot(grid[1, 0], polar = True)
BI = introfig.add_subplot(grid[1, 1], polar = True)
NEA = introfig.add_subplot(grid[1, 2], polar = True)

SEA = introfig.add_subplot(grid[2, 0], polar = True)
AL = introfig.add_subplot(grid[2, 1], polar = True)
FR = introfig.add_subplot(grid[2, 2], polar = True)

#bar plot
fire_occ = introfig.add_subplot(grid[1:3, 3])

#--------------------------------------------------------
#seasonality
reg_ax_list = [SC, WMD, EMD, IP, ME, BI, NEA, SEA, AL, FR]

reg_list =["SC", "WMD", "EMD", "IP", #spring
           "ME", "BI", "NEA",  #summer
           "SEA", "AL", "FR"   #mixed
          ]

sig_list = {
    'MAM': ['SC', 'BI', 'ME', 'NEA', 'SEA', 'WMD', 'EMD', 'FR', 'IP'],
    'JJA': ['AL', 'SEA', 'IP', 'WMD', 'EMD'],
    'SON': ['FR', 'AL', 'WMD', 'SEA', 'EMD'],
    'DJF': ['BI', 'ME', 'SEA', 'NEA', 'EMD']
}

region_ind = 0

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
lb = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)', '(i)', '(j)', '(k)', '(l)']

for (ax_ind, ax) in enumerate(reg_ax_list):
    
    reg = reg_list[region_ind]  
    
    #get data and angles
    fire_seas_reg = fire_seas[fire_seas["Region"] == reg]
    num_ev = list(fire_seas_reg["EV"])
    burned_area = list(fire_seas_reg["BA"])
    angles = np.linspace(0, 2 * np.pi, len(months), endpoint=False).tolist()
    
    # close the loop, append the first to the end of the list
    num_ev += num_ev[:1]
    burned_area += burned_area[:1]
    angles += angles[:1]
    
    # Draw one axe per variable and add labels
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1], months, fontsize = 16)
    ax.tick_params(axis='x', pad=8)
    
    # Draw ylabels
    ax.set_rlabel_position(0)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0], ["0", "0.2", "0.4", "0.6", "0.8", "1.0"], color="black", fontsize=12)
    ax.set_ylim(0, 1)

    #Highlight synchronicity
    for season in ['MAM', 'JJA', 'SON', 'DJF']:
        start_angle = angles[2] if season == 'MAM' else angles[5] if season == 'JJA' else angles[8] if season == 'SON' else angles[11]
        end_angle = angles[4] if season == 'MAM' else angles[7] if season == 'JJA' else angles[10] if season == 'SON' else angles[1]


        if reg in sig_list[season]:
            if season != 'DJF':
                arc_theta = np.linspace(start_angle, end_angle, 100)
                r = np.ones_like(arc_theta)
                ax.plot(arc_theta, r, color = 'red', linewidth = 5, linestyle = '-')
            else:
                arc_theta = np.linspace(start_angle, 2*np.pi, 100)
                r = np.ones_like(arc_theta)
                ax.plot(arc_theta, r, color = 'red', linewidth = 5, linestyle = '-')

                arc_theta = np.linspace(0, end_angle, 100)
                r = np.ones_like(arc_theta)
                ax.plot(arc_theta, r, color = 'red', linewidth = 5, linestyle = '-')
    
    # Plot data
    ax.plot(angles, burned_area, color='#33A036', linewidth=2, linestyle='solid')
    ax.fill(angles, burned_area, color='#33A036', alpha=0.25)
    
    ax.plot(angles, num_ev, color='blue', linewidth=2, linestyle='solid')
    ax.fill(angles, num_ev, color='blue', alpha=0.25)

    ax.set_title('NEE' if reg == 'NEA' else 'SEE' if reg == 'SEA' else 'CE' if reg == 'ME' else reg, fontsize = 24) #adjust region abbreviations
    ax.text(-0.15, 1.15, lb[ax_ind], transform = ax.transAxes, fontsize = 25)

    region_ind += 1


#--------------------------------------------------------
#sum fire occurrences and burned area for each region
ba = pd.DataFrame(fire_obs.groupby("region")["area"].sum()).reset_index()
occ = pd.DataFrame(fire_obs["region"].value_counts()).reset_index()
df = pd.merge(ba, occ, on = "region")
df = df.sort_values(by = "count")
df = df.rename(columns = {"count": "occ", "area": "ba"})
tick_label_dict = dict(zip([i for i in range(10)],df['region']))
tick_label_dict.update({
    1: 'CE',
    6: 'SEE',
    8: 'NEE',                    
})  #adjust region abbreviations

fire_occ.barh([i+0.17 for i in range(10)], df["occ"], height = 0.3, facecolor = "blue", edgecolor = "none", linewidth = 0, alpha = 0.25)
fire_occ.barh([i+0.17 for i in range(10)], df["occ"], height = 0.3, facecolor = "none", edgecolor = "blue", linewidth = 2)
fire_occ.spines['left'].set_visible(False)
fire_occ.spines['right'].set_visible(False)
fire_occ.set_yticks([i for i in range(10)], [tick_label_dict[i] for i in range(10)])
fire_occ.tick_params(axis = "y", length = 0, labelsize = 18)
fire_occ.tick_params(axis = "x", labelsize = 18)
fire_occ.set_xlabel("Fire event number (-)", fontsize = 19)

fire_occ_tx = fire_occ.twiny()
fire_occ_tx.barh([i-0.17 for i in range(10)], df["ba"], height = -0.3, facecolor = "#33A036", edgecolor = "none", linewidth = 0, alpha = 0.25)
fire_occ_tx.barh([i-0.17 for i in range(10)], df["ba"], height = -0.3, facecolor = "none", edgecolor = "#33A036", linewidth = 2)
fire_occ_tx.spines["left"].set_visible(False)
fire_occ_tx.spines["right"].set_visible(False)
fire_occ_tx.tick_params(axis = "x", labelsize = 18)
fire_occ_tx.set_xlabel(f"Burned area ($km^{2}$)", fontsize = 19, labelpad = 10)

fire_occ.set_box_aspect(2.3)
fire_occ.text(-0.2, 1.08, '(k)', transform = fire_occ.transAxes, fontsize = 25)

#legend
blue_square = mlines.Line2D([], [], marker = "s", markersize = 25, markerfacecolor = (0, 0, 1, 0.25), markeredgecolor = "blue", markeredgewidth = 2, linestyle = "None", label = "Fire event number")
green_square = mlines.Line2D([], [], marker = "s", markersize = 25, markerfacecolor =  ('#33A036', 0.25), markeredgecolor = "#33A036", markeredgewidth = 2, linestyle = "None", label = "Burned area")
fire_occ.legend(handles = [blue_square, green_square], 
                loc='upper left', fontsize=18, title='', frameon = False, ncol = 1, bbox_to_anchor=(-0.05, 0.18), handletextpad = 0, labelspacing = 1)

#--------------------------------------------------------
introfig.subplots_adjust(wspace=0.4, hspace=0.3)

pos = fire_occ.get_position()
new_bottom = pos.y0 + 0.01  
fire_occ.set_position([pos.x0, new_bottom, pos.width, pos.height])

#save
#introfig.savefig("/home/lixinh/SynFire_Scripts/WP1/Figures/v5_2_seasonality.png", dpi = 600, bbox_inches = "tight")
introfig.show()